# Demo 1: KV Cache 显存分析与推理延迟分解

本 notebook 包含：
1. KV Cache 显存占用柱状图 → `assets/chart-kv-memory.png`
2. 推理延迟瀑布图 → `assets/chart-latency-breakdown.png`
3. Roofline Model 图 → `assets/chart-roofline.png`

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.font_manager as fm
import numpy as np

# 注册中文字体
fm.fontManager.addfont('/home/zhangboju/.fonts/NotoSansSC.ttf')

# 使用 Ocean Depths 配色
matplotlib.rcParams.update({
    'figure.facecolor': '#1a2332',
    'axes.facecolor': '#1a2332',
    'axes.edgecolor': '#8faab0',
    'axes.labelcolor': '#f1faee',
    'text.color': '#f1faee',
    'xtick.color': '#8faab0',
    'ytick.color': '#8faab0',
    'grid.color': '#223044',
    'legend.facecolor': '#223044',
    'legend.edgecolor': '#8faab0',
    'font.size': 12,
    'font.family': 'Noto Sans SC',
    'axes.unicode_minus': False,
})

## 1. KV Cache 显存占用柱状图

In [ ]:
def calc_kv_cache_memory(n_layers, d_model, seq_len, bytes_per_elem=2):
    """计算 KV Cache 显存占用 (bytes)"""
    return 2 * n_layers * d_model * seq_len * bytes_per_elem

# 模型配置
models = {
    'LLaMA-2-7B': {'n_layers': 32, 'd_model': 4096},
    'LLaMA-2-70B': {'n_layers': 80, 'd_model': 8192},
}

seq_lengths = [512, 1024, 2048, 4096, 8192]

fig, ax = plt.subplots(figsize=(10, 5.5))

x = np.arange(len(seq_lengths))
width = 0.35

for i, (name, cfg) in enumerate(models.items()):
    mem_gb = [calc_kv_cache_memory(cfg['n_layers'], cfg['d_model'], s) / 1e9
              for s in seq_lengths]
    color = '#2d8b8b' if '7B' in name else '#e0b060'
    bars = ax.bar(x + i * width, mem_gb, width, label=name, color=color, alpha=0.85,
                  edgecolor='none')
    for bar, val in zip(bars, mem_gb):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}', ha='center', va='bottom', fontsize=9, color='#a8dadc')

# GPU memory limits
ax.axhline(y=24, color='#e09060', linestyle='--', alpha=0.7, linewidth=1.5)
ax.text(len(seq_lengths)-1 + 0.7, 24, 'A10G 24GB', fontsize=10, color='#e09060', va='center')
ax.axhline(y=80, color='#e09060', linestyle='--', alpha=0.7, linewidth=1.5)
ax.text(len(seq_lengths)-1 + 0.7, 80, 'A100 80GB', fontsize=10, color='#e09060', va='center')

ax.set_xlabel('序列长度 (tokens)')
ax.set_ylabel('KV Cache 显存 (GB)')
ax.set_title('KV Cache 显存占用 vs 序列长度 (FP16)', fontsize=14, color='#a8dadc', pad=12)
ax.set_xticks(x + width / 2)
ax.set_xticklabels([str(s) for s in seq_lengths])
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 110)

plt.tight_layout()
plt.savefig('../assets/chart-kv-memory.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: chart-kv-memory.png')

## 2. 推理延迟瀑布图

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4.5), sharey=True)

output_lengths = [100, 500, 1000]
ttft = 0.15  # Time to first token (seconds)
tpot = 0.025  # Time per output token (seconds)

for ax, n_out in zip(axes, output_lengths):
    total_decode = n_out * tpot

    # Prefill bar
    ax.barh(0, ttft, height=0.6, color='#2d8b8b', edgecolor='none')
    ax.text(ttft/2, 0, f'Prefill\n{ttft*1000:.0f}ms',
            ha='center', va='center', fontsize=9, color='#1a2332', fontweight='bold')

    # Decode bars (show as stacked segments)
    n_segments = min(n_out, 10)
    seg_size = total_decode / n_segments
    left = ttft
    for j in range(n_segments):
        color = '#5ec4c4' if j < n_segments - 1 else '#a8dadc'
        ax.barh(0, seg_size, left=left, height=0.6, color=color, edgecolor='none', alpha=0.7)
        left += seg_size
    ax.text(ttft + total_decode/2, 0, f'Decode\n{total_decode:.1f}s ({n_out} tokens)',
            ha='center', va='center', fontsize=9, color='#1a2332', fontweight='bold')

    total = ttft + total_decode
    ax.set_title(f'输出 {n_out} tokens\n总计 {total:.1f}s', fontsize=11, color='#a8dadc')
    ax.set_xlim(0, 28)
    ax.set_xlabel('时间 (秒)', fontsize=10)

# Annotations
fig.suptitle('推理延迟分解: Prefill + N × Decode', fontsize=13, color='#a8dadc', y=1.02)
plt.tight_layout()
plt.savefig('../assets/chart-latency-breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: chart-latency-breakdown.png')

## 3. Roofline Model 图

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

# Roofline parameters (simplified for A100)
peak_bandwidth = 2000  # GB/s (HBM)
peak_flops = 312e12    # TFLOP/s (FP16 with Tensor Core)

ai = np.logspace(-2, 3, 500)  # Arithmetic Intensity (FLOP/Byte)

# Memory-bound region: performance = bandwidth * AI
perf_mem = peak_bandwidth * 1e9 * ai  # FLOP/s
# Compute-bound region: performance = peak_flops
perf_compute = np.full_like(ai, peak_flops)
# Ridge point
ridge_ai = peak_flops / (peak_bandwidth * 1e9)

perf = np.minimum(perf_mem, perf_compute)

ax.loglog(ai, perf / 1e12, color='#a8dadc', linewidth=2.5)
ax.fill_between(ai, perf / 1e12, alpha=0.08, color='#a8dadc')

# Annotate regions
ax.text(0.15, 2.5, 'Memory-bound\n(瓶颈在带宽)', fontsize=12, color='#e09060', fontweight='bold')
ax.text(80, 350, 'Compute-bound\n(瓶颈在算力)', fontsize=12, color='#5ec4c4', fontweight='bold')

# Mark typical operations
ops = {
    'Decode\nAttention': (0.05, 0.1),
    'Decode\nFFN': (0.5, 1.5),
    'Prefill\nFFN': (50, 200),
}
for name, (x, y) in ops.items():
    color = '#e09060' if 'Decode' in name else '#5ec4c4'
    ax.plot(x, y, 'o', markersize=10, color=color, zorder=5)
    offset = (15, 12) if 'Decode' in name else (15, -20)
    ax.annotate(name, (x, y), textcoords='offset points', xytext=offset,
                fontsize=9, color=color, fontweight='bold')

# Ridge point
ax.axvline(x=ridge_ai, color='#8faab0', linestyle=':', alpha=0.5)
ax.text(ridge_ai, 400, f'Ridge Point\nAI={ridge_ai:.0f}', fontsize=9,
        color='#8faab0', ha='center')

ax.set_xlabel('算术强度 (FLOP / Byte)', fontsize=12)
ax.set_ylabel('性能 (TFLOP/s)', fontsize=12)
ax.set_title('Roofline Model — LLM 推理瓶颈分析 (A100)', fontsize=13, color='#a8dadc', pad=10)
ax.set_xlim(0.01, 1000)
ax.set_ylim(0.01, 500)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('../assets/chart-roofline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: chart-roofline.png')